# 5 (Capstone) - The Base: Twenty Labeled Cases

Chapter 5 derives a taxonomy of base question types from the graph and gives three sources that turn a category into concrete items. This companion shows the capstone's base: **twenty labeled complaint cases**, the experimental material the whole campaign is built on.

Each case instantiates a taxonomy category, and each carries *per-component* ground truth --- not one label but the expected classification, product, issue, governing policy and escalation decision. That per-component key is what lets a later chapter test the agent's trajectory and attribute a failure to a single tool, rather than reading only a final right/wrong bit.

## The cases enter as labeled seed cases

Of the three base sources in Chapter 5 --- user-supplied pairs, labeled seed cases, and store-mined generators --- the capstone uses the **second**. The cases are hand-labeled (`data/eval_cases/cases.json`) and enter the framework through `SeedCaseSource`, the source built for trajectory testing: each item's `components` field carries the per-tool truth the agent is graded against step by step.

One point worth fixing in place: all three sources are **user-selected**. The difference is not "hand-written vs automatic" but *where the questions and answers come from* --- you author them (seed cases, user pairs) or you name the question types and let the generators read answers from the graph (store-mined). Even the store-mined path does not pick categories for you; you choose them, and a chosen category simply yields nothing when the graph has no structure to fill it. Notebook 5b works that third source through on the policy store.

**What the next cell does** — loads the twenty hand-labeled cases and turns them into framework base items:

1. **Import.** Imports the open `proofloop.suite` package as `gt` (the catalog-driven test-suite builder).
2. **Locate the data.** Walks parent directories from the cwd until it finds `data/eval_cases/cases.json`, then reads and JSON-parses it into `cases`.
3. **Build items.** Wraps the raw dicts in `gt.SeedCaseSource(cases)` and calls `.items()`, which emits one `QAItem` per case with `base='seed_case'`, the `message` text as `query` and the per-component fields collected into `components`.
4. **Inspect two.** Indexes the items by `qid` and prints `case-001` (a benign complaint, `expected_escalation` False) and `case-016` (an adversarial one, `expected_escalation` True) with their full component keys.

In [ ]:
import pathlib, json
import proofloop.suite as gt

# labeled seed cases are bundled in this repo's data/ (see data/eval_cases/)
root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / 'data' / 'eval_cases' / 'cases.json').exists())
cases = json.loads((root / 'data' / 'eval_cases' / 'cases.json').read_text())
items = gt.SeedCaseSource(cases).items()
print(len(items), 'labeled seed cases\n')

by_id = {it.qid: it for it in items}
for qid in ('case-001', 'case-016'):           # a benign complaint and an adversarial one
    it = by_id[qid]
    print(f'{it.qid}: {it.query}')
    print(f'   components: {it.components}\n')

## Which taxonomy category each case instantiates

The twenty cases populate two of Chapter 5's categories. A **`conditional_rule`** case is a policy-conditioned complaint or inquiry: its correct handling depends on a policy threshold, so its ground truth is the calibrated rule the graph records. A **`governance_policy`** case is adversarial --- a PII message, a prompt-injection attempt or a high-severity regulatory complaint --- and its ground truth is the expected *behavior* (refuse or escalate), not a fact in the graph. The case's `factors` mark which it is.

**What the next cell does** — classifies each case into its taxonomy category and counts the split:

1. **Define the rule.** `base_category(case)` reads the case's `factors` and calls a case adversarial when `user_intent == 'adversarial'` or `regulatory != 'none'`, mapping those to `governance_policy` and everything else to `conditional_rule`.
2. **Count categories.** Runs the rule over every case through a `Counter` into `split`, then prints the per-category totals.
3. **Count adversarial markers.** Tallies the `regulatory` factor across cases and prints every non-`none` marker (PII, Reg_X, UDAAP, prompt_injection) with its count.

In [ ]:
from collections import Counter

def base_category(case):
    f = case.get('factors', {})
    adversarial = f.get('user_intent') == 'adversarial' or f.get('regulatory', 'none') != 'none'
    return 'governance_policy' if adversarial else 'conditional_rule'

split = Counter(base_category(c) for c in cases)
print('taxonomy category split:')
for cat, n in sorted(split.items()):
    print(f'  {cat:18s} {n}')

print('\nadversarial markers (the regulatory factor on governance cases):')
for marker, n in sorted(Counter(c['factors'].get('regulatory', 'none') for c in cases).items()):
    if marker != 'none':
        print(f'  {marker:18s} {n}')

## Per-component ground truth

A single end-to-end label would tell you the agent failed but not *where*. The seed cases instead carry a key for every step the agent runs --- classification, product, issue, governing policy, escalation --- so each tool can be scored on its own and a system failure localized to the first tool that erred (the weak-link analysis of Chapter 10). Every case carries the full set.

**What the next cell does** — verifies the per-component ground truth is complete and reads two label distributions:

1. **Name the keys.** Lists the five scored fields in `COMPONENTS` (`expected_classification`, `expected_product`, `expected_issue`, `expected_policy`, `expected_escalation`).
2. **Check coverage.** For each field, counts how many of the twenty raw `cases` carry it and prints the `present in N/20` line.
3. **Read distributions.** Prints the `Counter` of `expected_classification` values and the number of cases with `expected_escalation` True, both pulled straight from the ground-truth key.

In [ ]:
COMPONENTS = ['expected_classification', 'expected_product', 'expected_issue',
              'expected_policy', 'expected_escalation']
print('per-component ground-truth coverage:')
for fld in COMPONENTS:
    print(f'  {fld:26s} present in {sum(fld in c for c in cases)}/{len(cases)}')

print('\nlabel distributions read from the key:')
print('  classification :', dict(Counter(c['expected_classification'] for c in cases)))
print('  escalate=True  :', sum(c['expected_escalation'] for c in cases),
      'of', len(cases))

These twenty cases are the base the rest of the campaign builds on. Chapter 6 enriches each into many phrasings without changing its answer (notebook 12.3), and every component label here is scored against the graph-read answer key of Chapter 4 (notebook 12.4), so a failure is chargeable to the agent rather than to the key.

## Extending the base

The two categories above are what *these* twenty cases happen to instantiate --- not a ceiling. A user can specify more in three ways, and the framework imposes no limit:

1. **More categories.** Add cases that instantiate other taxonomy categories. Here we add a `recovery` case --- an ill-formed, unanswerable message whose correct response is to *abstain* rather than fabricate (the behavioral category of Section 5.5).
2. **More per-component truth.** `SeedCaseSource` takes a configurable `component_keys`; pass your own to capture a new per-tool field (here `expected_behavior`). Anything not a component, query or id is still carried in `metadata`.
3. **More sources, one campaign.** The enrichment layer treats all three base sources identically, so seed cases compose with bring-your-own `(query, answer)` pairs (`UserBaseSource`, no graph) or store-mined graph questions (`CatalogBaseSource`) in a single suite.

**What the next cell does** — extends the base in the three ways just described:

1. **New category.** Builds `recovery_case`, an ill-formed `'asdkfj ??? help'` message whose correct response is to abstain, carrying a new `expected_behavior` field the default key set does not name.
2. **Custom component key.** Calls `gt.SeedCaseSource(...)` over `cases + [recovery_case]` with `component_keys` extended by `'expected_behavior'`, so `.items()` picks the new field up as scored ground truth; prints the last item to show `expected_behavior` landed in `components`.
3. **Compose sources.** Builds a bring-your-own `(query, answer)` pair through `gt.UserBaseSource(...).items()` (which sets `base=None`, no graph), concatenates it onto the seed items as `mixed`, and prints the combined count and the user item's `base` to show all sources compose in one suite.

In [ ]:
# (1) a new category: a recovery case (ill-formed/unanswerable -> correct behavior is abstain),
#     carrying a NEW per-component field the default key set does not name.
recovery_case = {
    'id': 'case-021',
    'message': 'asdkfj ??? help',
    'expected_classification': 'other',
    'expected_escalation': False,
    'expected_behavior': 'abstain',                 # <-- new per-tool ground truth
    'factors': {'user_intent': 'ambiguous', 'regulatory': 'none'},
}

# (2) a custom component_keys picks the new field up as scored ground truth
extended = gt.SeedCaseSource(
    cases + [recovery_case],
    component_keys=tuple(COMPONENTS) + ('expected_behavior',),
).items()
new = extended[-1]
print('added category : recovery (behavioral -- recognized by the abstain behavior)')
print('new case       :', new.qid, '|', new.query)
print('components     :', new.components, '   <- carries expected_behavior')

# (3) compose seed cases with a bring-your-own pair (no graph); all sources are treated alike
byo = gt.UserBaseSource([
    {'query': 'What is the overdraft fee?', 'answer': '35'},   # an exact_recall-style pair
]).items()
mixed = extended + byo
print(f'\nmixed suite    : {len(extended)} seed + {len(byo)} user = {len(mixed)} base items')
print('user item base :', byo[0].base, '(None -> the supplied answer IS the ground truth)')

## Self-check

The asserts below pin every number this notebook claims --- twenty seed cases, a ten/ten split across the two taxonomy categories, eight adversarial markers and full per-component coverage. Run the notebook end to end and it fails loud if any of these drift, so the illustration stays reproducible rather than quietly going stale.

**What the next cell does** — asserts every number the notebook claims so the illustration fails loud if it drifts:

1. **Source and count.** Checks there are twenty `items` and every one has `base == 'seed_case'`.
2. **Category split.** Asserts ten `conditional_rule` and ten `governance_policy` in `split`, and eight cases with a non-`none` `regulatory` marker.
3. **Coverage and escalation.** Asserts every case carries all five `COMPONENTS`, that `case-001`'s `expected_escalation` is False and `case-016`'s is True, then prints the OK line.

In [ ]:
assert len(items) == 20 and all(it.base == 'seed_case' for it in items)   # labeled seed-case source
assert split['conditional_rule'] == 10 and split['governance_policy'] == 10
assert sum(c['factors'].get('regulatory', 'none') != 'none' for c in cases) == 8   # adversarial markers
assert all(all(fld in c for fld in COMPONENTS) for c in cases)            # full per-component key
assert by_id['case-001'].components['expected_escalation'] is False       # benign complaint, no escalation
assert by_id['case-016'].components['expected_escalation'] is True        # UDAAP, must escalate
print('OK - Ch5 capstone: twenty labeled cases, two categories, full per-component ground truth')